# Urban and rural surrounding areas selection

***
> This work is licensed under a [Creative Commons Attribution 4.0 International License](http://creativecommons.org/licenses/by/4.0)
>
> ![Creative Commons License >](https://i.creativecommons.org/l/by/4.0/88x31.png)

Adapted from the URCLIMASK example notebook (https://github.com/FPS-URB-RCC/URCLIMASK, CC BY 4.0).
The URCLIMASK code was adapted to read simulation data on a Lambert conformal grid and our own urban station data.

**Role in the paper.** For one city, this notebook computes the six p2p and a2a evaluation methods for the
nighttime UHI (UHIN = urban minus rural daily minimum temperature, monthly means):
1. builds the URCLIMASK urban/rural masks (a2a) from the static model fields;
2. classifies the stations as urban/rural with the urban centre (UCDB) and with LCZ;
3. computes simulated and observed UHIN per method, adds the p2a result of
   `method_2_blended_approach/UHI_blended_approach_<City>.ipynb` (run that notebook first) and saves all seven methods to
   `results/data/<City>/UHI_<City>_annualcycle_method_comparison_<var>.{nc,csv}`.

| Code name (columns / variables) | Paper name |
|---|---|
| `p2p_un_ucdb` | p2p_UCN_uc |
| `p2p_un_lcz` | p2p_UCN_lcz |
| `p2p_ms_ucdb` | p2p_MS_uc |
| `urclim_un_ucdb` | a2a_UCN_uc |
| `urclim_un_lcz` | a2a_UCN_lcz |
| `urclim_ms_ucdb` | a2a_MS_uc |
| `blend_ms_ucdb` (method 2, blended) | p2a_MS_dist |

## Load libraries

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import glob
import numpy as np
import xarray as xr
import cf_xarray as cfxr
from pathlib import Path
import metobs_toolkit

from rasterio import warp

import cftime
import pandas as pd
import matplotlib as plt
from pprint import pprint
import importlib
import cartopy.crs as ccrs
import geopandas as gpd
import rasterio
import sys

from urclimask.urban_areas import (
    UrbanVicinity
)
from urclimask.utils import (
    kelvin2degC,
    traverseDir,
    fix_360_longitudes,
    RCM_DICT,
    load_ucdb_city,
    align_to_reference_grid,
    classify_stations_from_ucdb_city,
    plot_UHI_p2p_cycle,
    compute_UHI_p2p, 
    from_latlon_to_xy,
    classify_stations_from_LCZ
)

from urclimask.UHI_analysis import (
    UrbanIsland
)
from urclimask.GHCNd_stations import (
    load_ghcnd_stations,
    get_ghcnd_df,
    get_valid_timeseries, 
    available_vars,
    inside_city
)
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


# Urclimask

## Urclimask settings

In [ ]:
city = 'Zurich'
lon_city = 8.54144167156696
lat_city = 47.37656400426996

network = 'combined_network'
subset = '_gapfree'
start = "2019-07-01"
stop = "2021-11-30"

#Ghent (51.05454277446639, 3.7249005076743438)
domain = 'EUROCITIES'
domain_resolution = 2.5

model = 'ALAROSFX'
scenario = "evaluation" 

urban_var = 'sfturf'

urban_th = 40
urban_sur_th = 10
orog_diff = 100
sftlf_th = 60
ratio_r2u = 2
min_city_size = 15
lon_lim = 0.3
lat_lim = 0.3

Create a directory to save the results for this particular city and model

In [ ]:
ROOT = Path.cwd().parent.parent.parent
data_dir =  ROOT / f'data/'
plot_dir =  ROOT / f'results/figures/{city}/method_2_urclimask_{network}'
os.makedirs(plot_dir, exist_ok=True)
save_masks_dir = ROOT / f'data/metadata/masks/'
os.makedirs(save_masks_dir, exist_ok=True)
save_dir =  ROOT / f'results/data/{city}/'
os.makedirs(save_dir, exist_ok=True)
base_filename = f'{city}-{domain}_ECMWF-ERA5_{scenario}_r1i1p1f1_{network}_{subset}'

## Define urban areas and vicinity

### Load city shapefile and static variables
Load static variables, such as urban fraction (sfturf), terrain elevation (orography) or land-sea fraction (sftlf).

In [ ]:
ucdb_city = load_ucdb_city(f'{data_dir}/metadata/UCDB_shapefiles', city)

static_input_path = f'{data_dir}/metadata/simulations/static_variables_eurocities.nc'
static_input = xr.open_dataset(static_input_path,engine='netcdf4')
static_input = fix_360_longitudes(static_input)

ds_sfturf, ds_orog, ds_sftlf = static_input, static_input, static_input


In [ ]:
#Add parameters to the function
URBAN = UrbanVicinity(
    urban_sur_th = urban_sur_th,
    orog_diff = orog_diff,
    sftlf_th = sftlf_th,
    ratio_r2u = ratio_r2u,
    min_city_size = min_city_size,
    lon_city = lon_city,
    lat_city = lat_city,
    lon_lim = lon_lim,
    lat_lim = lat_lim,
    model = model,
    domain = domain,
    urban_th = urban_th,
    urban_var = urban_var
)

#crop around the city
ds_sfturf = URBAN.crop_area_city(ds = ds_sfturf, res = domain_resolution)
ds_orog = URBAN.crop_area_city(ds = ds_orog, res = domain_resolution)
ds_sftlf = URBAN.crop_area_city(ds = ds_sftlf, res = domain_resolution)

#Define masks using the parameters above
sfturf_mask, sfturf_sur_mask, orog_mask, sftlf_mask = URBAN.define_masks(
    ds_sfturf = ds_sfturf, 
    ds_orog = ds_orog, 
    ds_sftlf = ds_sftlf,
)

### Define rural vicinity areas

We use an iterative algorithm to select the urban vicinity excluding masked areas

In [ ]:
urmask = URBAN.select_urban_vicinity(
    sfturf_mask = sfturf_mask, 
    orog_mask = orog_mask,
    sftlf_mask = sftlf_mask,
    sfturf_sur_mask = sfturf_sur_mask,
    debug = True,
    max_iterations=200,
)

We can plot the masks and the urban vs vicinity areas. The latter are are passed via the `urban_areas` parameter. They are hidden if set to `None`.

In [ ]:
fig = URBAN.plot_static_variables(ds_sfturf = ds_sfturf, 
                                  ds_orog = ds_orog, 
                                  ds_sftlf = ds_sftlf,
                                  sfturf_mask = sfturf_mask, 
                                  orog_mask = orog_mask, 
                                  sftlf_mask = sftlf_mask,
                                  urban_areas = urmask,
                                  ucdb_city = ucdb_city,
                                 composite = False) 
fig.savefig(f"{plot_dir}/urmask_{base_filename}_fx_.pdf", bbox_inches='tight')

In [ ]:
fig = URBAN.plot_static_variables(ds_sfturf = ds_sfturf, 
                                  ds_orog = ds_orog, 
                                  ds_sftlf = ds_sftlf,
                                  sfturf_mask = sfturf_mask, 
                                  orog_mask = orog_mask, 
                                  sftlf_mask = sftlf_mask,
                                  urban_areas = urmask,
                                  ucdb_city = ucdb_city,
                                 composite = True, )
fig.savefig(f"{plot_dir}/urmask_{base_filename}_fx.pdf", bbox_inches='tight')

Save these masks and urban fraction to NetCDF

In [ ]:

urmask.to_netcdf(f"{save_masks_dir}/urmask_{base_filename}_fx.nc")

## Load simulation data

In [ ]:
var='tasmin'


In [ ]:
path_monmean = f"{data_dir}/processed/simulations/monthly_means/{domain}_operational_cy43_{var}_monmean_*.nc"
da_var = xr.open_mfdataset(sorted(glob.glob(path_monmean)), engine="netcdf4", chunks="auto").rename({'tas':var}).sel(time=slice(start,stop)) 

ds_RCM_urb = URBAN.crop_area_city(ds = da_var, res = domain_resolution)
ds_RCM_urb["time"] = ds_RCM_urb.indexes["time"].to_period("M").to_timestamp()
ds_RCM_urb

In [ ]:
# Align static information to model data coordinates (in case their is a rounding error in the grid coordinates causing a mismatch in the grid)
urmask, alignment_status = align_to_reference_grid(urmask, ds_RCM_urb, coord_names=['x', 'y'])
print(alignment_status)

## Load Observational data

### Load urban climate network data

In [ ]:
start_o = start #because 2020, the year represented by the Urban network is not in the meteostations network
stop_o = stop

# load station observation data (preproccessed: monthly mean of daily data)
obs_path = f"{data_dir}/processed/observations/{city}/{network}/datasets/{subset}/{city}_{network}{subset}_monthlymean_daily{var[3:]}.nc"
ds_obs = xr.open_dataset(obs_path, engine="netcdf4", chunks="auto").sel(time=slice(start_o,stop_o))
obs_timeseries = ds_obs.to_dataframe().copy()

obs_timeseries = obs_timeseries.drop('C6_AWEL_Werdinsel', axis='columns') #permanent water obs
obs_timeseries = obs_timeseries.drop('E7', axis='columns') #permanent water obs

stations = obs_timeseries.columns.to_list()
print(f"{len(stations)} available stations: {stations}")

Prepare obs_attributes metadata for two urban/rural classfiction approaches: core based (ucdb polygon) and LCZ based

In [ ]:
pd_meta = pd.read_csv(f"{data_dir}/processed/observations/{city}/{network}/{city}_{network}{subset}_metadata.csv")
pd_meta.rename(columns={'name':'code'}, inplace=True)
obs_attributes = classify_stations_from_ucdb_city(pd_meta, ucdb_city)
obs_attributes = obs_attributes[obs_attributes["code"].isin(stations)]

obs_attributes_lczbased = classify_stations_from_LCZ(pd_meta)
obs_attributes_lczbased = obs_attributes_lczbased[obs_attributes_lczbased["code"].isin(stations)]
obs_attributes


In [ ]:
for station in obs_timeseries.columns:
    print(f"Variable: {station}, nan count: {obs_timeseries[station].isnull().sum().item()}")

### Load synoptic stations (national meteorological stations)

In [ ]:
network_m='meteo_stations'
subset_m= ''
start_m = start
stop_m = stop


# load station observation data (preproccessed: monthly mean of daily data)
obs_path_tasmin_m = f"{data_dir}/processed/observations/{city}/{network_m}/datasets/{subset_m}/{city}_{network_m}{subset_m}_monthlymean_daily{var[3:]}.nc"
ds_obs_tasmin_m = xr.open_dataset(obs_path_tasmin_m, engine="netcdf4", chunks="auto").sel(time=slice(start,stop))
obs_timeseries_m = ds_obs_tasmin_m.to_dataframe().copy()
#obs_timeseries.drop('UKE00105909', axis='columns')
stations_m = obs_timeseries_m.columns.to_list()
print(f"{len(stations_m)} available stations: {stations_m}")
pd_meta = pd.read_csv(f"{data_dir}/processed/observations/{city}/{network_m}/{city}_{network_m}_metadata.csv")
pd_meta.rename(columns={'name':'code'}, inplace=True)
obs_attributes_m = classify_stations_from_ucdb_city(pd_meta, ucdb_city)
obs_attributes_m = obs_attributes_m[obs_attributes_m["code"].isin(stations_m)]
obs_attributes_m

In [ ]:

for station in obs_timeseries_m.columns:
    print(f"station: {station}, nan count: {obs_timeseries_m[station].isnull().sum().item()}")


In [ ]:
latex_table = obs_attributes[["code", "LCZ", "lat", "lon", "inside_city"]].rename(columns={'code':'Station', 'LCZ':'LCZ class', 'lat':'Latitude', 'lon':'Longitude', 'inside_city':'Inside city (UCDB)'}).copy()
latex_table.to_latex(f"{save_dir}/obs_attributes_{city}_{network}_{subset}.tex", index=False )
latex_table_m = obs_attributes_m[["code", "LCZ", "lat", "lon", "inside_city"]].rename(columns={'code':'Station', 'LCZ':'LCZ class', 'lat':'Latitude', 'lon':'Longitude', 'inside_city':'Inside city (UCDB)'}).copy()
latex_table_m.to_latex(f"{save_dir}/obs_attributes_m_{city}_{network}_{subset}.tex", index=False)

## UHI Urclim analysis

In [ ]:
proj_dict = ds_RCM_urb.crs.attrs
plot_proj = ccrs.LambertConformal(central_latitude=proj_dict["latitude_of_projection_origin"], central_longitude=proj_dict["longitude_of_central_meridian"], standard_parallels=proj_dict["standard_parallel"], globe=ccrs.Globe(semimajor_axis=proj_dict["earth_radius"], semiminor_axis=proj_dict["earth_radius"]))
plot_proj

obs_attributes[["x", "y"]] = obs_attributes.apply( lambda row: from_latlon_to_xy(row["lat"], row["lon"], plot_proj), axis=1, result_type="expand")
obs_attributes_m[["x", "y"]] = obs_attributes_m.apply( lambda row: from_latlon_to_xy(row["lat"], row["lon"], plot_proj), axis=1, result_type="expand")
obs_attributes_lczbased[["x", "y"]]= obs_attributes_lczbased.apply( lambda row: from_latlon_to_xy(row["lat"], row["lon"], plot_proj), axis=1, result_type="expand")

for obs_attr, method in [(obs_attributes, 'UCN_uc'), (obs_attributes_lczbased, 'UCN_lcz'), (obs_attributes_m, 'MS_uc')]:
    p2p_mask = xr.full_like(urmask["urmask"], np.nan, dtype=float)
    # Loop over stations
    for _, station in obs_attr.iterrows():

        # Find nearest grid point
        ix = np.abs(urmask.x.values - station["x"]).argmin()
        iy = np.abs(urmask.y.values - station["y"]).argmin()

        # Assign inside_city value (0 or 1)
        p2p_mask.values[iy, ix] = station["inside_city"]
    urmask[f"p2p_{method}"] = p2p_mask
urmask


### UHI Urclim : urban network , ubcd classification

Paper method: **a2a_UCN_uc** (code: `urclim_un_ucdb`).

In [ ]:
UHI = UrbanIsland(
    ds=ds_RCM_urb[var],
    urban_vicinity=urmask,
    anomaly="abs",  # or "rel"
    obs_attributes=obs_attributes,
    obs_timeseries=obs_timeseries
)
report = UHI.check_obs_sim_overlap()

#calculate the annual cycle of the UHI for thobservations
UHI.compute_annual_cycle()
ds_anomaly = UHI.ds_annual_cycle

rural_anomaly = ds_anomaly.where(UHI.urban_vicinity['urmask'] == 0)
urban_anomaly = ds_anomaly.where(UHI.urban_vicinity['urmask'] == 1)

y_dim, x_dim = UHI._get_spatial_dim_names(ds_anomaly)       

urban_mean = urban_anomaly.mean(dim = [y_dim, x_dim]).compute()
rural_mean = rural_anomaly.mean(dim = [y_dim, x_dim]).compute()

uhi_urclim_un_ucdb_sim = urban_mean.values
uhi_urclim_un_ucdb_obs = UHI.obs_annual_cycle['urban_mean'].values


fig = UHI.plot_UI_map(
    city_name = city,
    ucdb_city = ucdb_city, 
)
fig.savefig(f"{plot_dir}/anamolymap_UN_UCDBclass_{var}.pdf", bbox_inches='tight')

fig = UHI.plot_UI_annual_cycle(
    ds_anomaly = ds_anomaly,
    obs_anomaly = UHI.obs_annual_cycle,
    percentiles = [5,25],
    gridcell_series = True, 
    city_name = city,
    vmin = -1, vmax =3.5)
fig.savefig(f"{plot_dir}/annualcycle_UN_UCDBclass_{var}.pdf", bbox_inches='tight')

### UHI Urclim: urban network, LCZ classification

Paper method: **a2a_UCN_lcz** (code: `urclim_un_lcz`).

In [ ]:
UHI_lcz = UrbanIsland(
    ds=ds_RCM_urb[var],
    urban_vicinity=urmask,
    anomaly="abs",  # or "rel"
    obs_attributes=obs_attributes_lczbased,
    obs_timeseries=obs_timeseries
)
report = UHI_lcz.check_obs_sim_overlap()

#calculate the annual cycle of the UHI for thobservations
UHI_lcz.compute_annual_cycle()
ds_anomaly = UHI_lcz.ds_annual_cycle

rural_anomaly = ds_anomaly.where(UHI_lcz.urban_vicinity['urmask'] == 0)
urban_anomaly = ds_anomaly.where(UHI_lcz.urban_vicinity['urmask'] == 1)

y_dim, x_dim = UHI_lcz._get_spatial_dim_names(ds_anomaly)       

urban_mean = urban_anomaly.mean(dim = [y_dim, x_dim]).compute()
rural_mean = rural_anomaly.mean(dim = [y_dim, x_dim]).compute()

uhi_urclim_un_lcz_sim = urban_mean.values
uhi_urclim_un_lcz_obs = UHI_lcz.obs_annual_cycle['urban_mean'].values


fig = UHI_lcz.plot_UI_map(
    city_name = city,
    ucdb_city = ucdb_city, 
)
fig.savefig(f"{plot_dir}/anamolymap_UN_LCZclass_{var}", bbox_inches='tight')

fig = UHI_lcz.plot_UI_annual_cycle(
    ds_anomaly = ds_anomaly,
    obs_anomaly = UHI_lcz.obs_annual_cycle,
    percentiles = [5,25],
    gridcell_series = True, 
    city_name = city,
    vmin = -1, vmax =3.5)
fig.savefig(f"{plot_dir}/annualcycle_UN_LCZclass_{var}.pdf", bbox_inches='tight')

### UHI urclim, national meteorological stations, UBCD classificatino

Paper method: **a2a_MS_uc** (code: `urclim_ms_ucdb`).

In [ ]:
UHI_m = UrbanIsland(
    ds=ds_RCM_urb[var],
    urban_vicinity=urmask,
    anomaly="abs",  # or "rel"
    obs_attributes=obs_attributes_m,
    obs_timeseries=obs_timeseries_m
)
report = UHI_m.check_obs_sim_overlap()

#calculate the annual cycle of the UHI for thobservations
UHI_m.compute_annual_cycle()
ds_anomaly = UHI_m.ds_annual_cycle

rural_anomaly = ds_anomaly.where(UHI_m.urban_vicinity['urmask'] == 0)
urban_anomaly = ds_anomaly.where(UHI_m.urban_vicinity['urmask'] == 1)

y_dim, x_dim = UHI_m._get_spatial_dim_names(ds_anomaly)       

urban_mean = urban_anomaly.mean(dim = [y_dim, x_dim]).compute()
rural_mean = rural_anomaly.mean(dim = [y_dim, x_dim]).compute()

uhi_urclim_ms_sim = urban_mean.values
uhi_urclim_ms_obs = UHI_m.obs_annual_cycle['urban_mean'].values


fig = UHI_m.plot_UI_map(
    city_name = city,
    ucdb_city = ucdb_city, 
)
fig.savefig(f"{plot_dir}/anamolymap_MS_UCDBclass_{var}", bbox_inches='tight')

fig = UHI_m.plot_UI_annual_cycle(
    ds_anomaly = ds_anomaly,
    obs_anomaly = UHI_m.obs_annual_cycle,
    percentiles = [5,25],
    gridcell_series = True, 
    city_name = city,
    vmin = -1, vmax =3.5)

fig.savefig(f"{plot_dir}/annualcycle_MS_UCDBclass_{var}.pdf", bbox_inches='tight')

# Point to point analysis 

In [ ]:
proj_dict = ds_RCM_urb.crs.attrs
plot_proj = ccrs.LambertConformal(central_latitude=proj_dict["latitude_of_projection_origin"], central_longitude=proj_dict["longitude_of_central_meridian"], standard_parallels=proj_dict["standard_parallel"], globe=ccrs.Globe(semimajor_axis=proj_dict["earth_radius"], semiminor_axis=proj_dict["earth_radius"]))
plot_proj

Paper method **p2p_UCN_uc**: each station is compared with its own grid cell; UHIN = mean(urban) - mean(rural), stations classified with the UCDB urban centre.

In [ ]:
fig,ax = plt.subplots(figsize=(10, 6))
uhi_p2p_un_core= compute_UHI_p2p( obs_attributes, obs_timeseries, ds_RCM_urb, proj=plot_proj,mean=False, variable=var)
uhi_p2p_un_ucdb_obs, uhi_p2p_un_ucdb_sim = plot_UHI_p2p_cycle( ax, uhi_p2p_un_core, mean=False,  percentiles=[20], plot_individual=False)
plt.show()

Paper method **p2p_UCN_lcz**: as above, stations classified with LCZ (urban: LCZ 1-10, rural: LCZ A-G).

In [ ]:
fig,ax = plt.subplots(figsize=(10, 6))
uhi_p2p_un_lcz = compute_UHI_p2p( obs_attributes_lczbased, obs_timeseries, ds_RCM_urb, proj=plot_proj,mean=False, variable=var)
uhi_p2p_un_lcz_obs, uhi_p2p_un_lcz_sim = plot_UHI_p2p_cycle( ax, uhi_p2p_un_lcz, mean=False,  percentiles=[20], plot_individual=False)
plt.show()

Paper method **p2p_MS_uc**: as above, with the meteorological service stations.

In [ ]:
uhi_un_p2p_fig,ax = plt.subplots(figsize=(10, 6))
uhi_p2p_ms = compute_UHI_p2p( obs_attributes_m, obs_timeseries_m, ds_RCM_urb, proj=plot_proj,mean=False, variable=var)
uhi_p2p_ms_obs, uhi_p2p_ms_sim = plot_UHI_p2p_cycle( ax, uhi_p2p_ms, mean=False,  percentiles=[20], plot_individual=False)
plt.show()

In [ ]:

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': plot_proj})
ax = UHI.plot_UI_urclim_visualisation(proj = plot_proj, ax=ax, mask_name = 'p2p_UCN_uc', ucdb_city = ucdb_city)
plt.show()

# Combine and save P2P, URCLIM and Blended data in dataset and dataframe

Adds the p2a result (`blend`) from the blended-approach notebook and saves the seven methods (sim, obs, bias per month) to `results/data/<City>/`.

In [ ]:
sim_ts = xr.open_dataset(f"{save_dir}/method_2_blended_approach\{city}_UHI_method2_simulation.nc", engine="netcdf4")
obs_ts = xr.open_dataset(f"{save_dir}/method_2_blended_approach\{city}_UHI_method2_observations.nc", engine="netcdf4")

uhi_blend_obs = obs_ts[f'UHI_{var}'].groupby('time.month').mean().values
uhi_blend_sim = sim_ts[f'UHI_{var}'].groupby('time.month').mean().values

In [ ]:
if var=='tasmin':
    suffix = 'N'
elif var=='tasmax':
    suffix = 'X'

#store all biases in a dataset 
uhis = xr.Dataset(
    {   f'UHI{suffix}_p2p_un_ucdb_obs': (["month"], uhi_p2p_un_ucdb_obs),
        f'UHI{suffix}_p2p_un_ucdb_sim': (["month"], uhi_p2p_un_ucdb_sim),
        f'UHI{suffix}_p2p_un_ucdb_bias': (["month"], uhi_p2p_un_ucdb_sim- uhi_p2p_un_ucdb_obs),

        f'UHI{suffix}_p2p_un_lcz_obs': (["month"], uhi_p2p_un_lcz_obs),
        f'UHI{suffix}_p2p_un_lcz_sim': (["month"], uhi_p2p_un_lcz_sim),
        f'UHI{suffix}_p2p_un_lcz_bias': (["month"], uhi_p2p_un_lcz_sim- uhi_p2p_un_lcz_obs),


        f'UHI{suffix}_p2p_ms_ucdb_obs': (["month"], uhi_p2p_ms_obs),
        f'UHI{suffix}_p2p_ms_ucdb_sim': (["month"], uhi_p2p_ms_sim),
         f'UHI{suffix}_p2p_ms_ucdb_bias': (["month"], uhi_p2p_ms_sim- uhi_p2p_ms_obs),


        #-----------------------------------------------------------
        f'UHI{suffix}_urclim_un_ucdb_obs': (["month"], uhi_urclim_un_ucdb_obs),
        f'UHI{suffix}_urclim_un_ucdb_sim': (["month"], uhi_urclim_un_ucdb_sim),
        f'UHI{suffix}_urclim_un_ucdb_bias': (["month"], uhi_urclim_un_ucdb_sim- uhi_urclim_un_ucdb_obs),

        f'UHI{suffix}_urclim_un_lcz_obs': (["month"], uhi_urclim_un_lcz_obs),
        f'UHI{suffix}_urclim_un_lcz_sim': (["month"], uhi_urclim_un_lcz_sim),
        f'UHI{suffix}_urclim_un_lcz_bias': (["month"], uhi_urclim_un_lcz_sim- uhi_urclim_un_lcz_obs),


        f'UHI{suffix}_urclim_ms_ucdb_obs': (["month"], uhi_urclim_ms_obs),
        f'UHI{suffix}_urclim_ms_ucdb_sim': (["month"], uhi_urclim_ms_sim),
        f'UHI{suffix}_urclim_ms_ucdb_bias': (["month"], uhi_urclim_ms_sim- uhi_urclim_ms_obs),

        #-----------------------------------------------------------
        f'UHI{suffix}_blend_ms_ucdb_obs': (["month"], uhi_blend_obs),
        f'UHI{suffix}_blend_ms_ucdb_sim': (["month"], uhi_blend_sim),
        f'UHI{suffix}_blend_ms_ucdb_bias': (["month"], uhi_blend_sim- uhi_blend_obs),

    },
    coords={
        "month": np.arange(1, 13)
    }
)
uhis.to_netcdf(f"{save_dir}/UHI_{city}_annualcycle_method_comparison_{var}.nc")
uhis_df = uhis.to_dataframe()
uhis_df.to_csv(f"{save_dir}/UHI_{city}_annualcycle_method_comparison_{var}.csv")
uhis_df

## Diagnostic plots
Annual cycles of simulated, observed UHIN and bias for the seven methods (not paper figures; Fig. 2 is made in `scripts/plotting/fig2_bias_spread_rennes.ipynb`).

In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label=f'UHI{suffix}_urclim_un_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label=f'UHI{suffix}_urclim_un_lcz_obs', color='peru', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label=f'UHI{suffix}_urclim_ms_ucdb_obs', color='red', linestyle='--', marker='o', lw=4)


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'], color='darkred', linestyle=(0,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'], color='peru', linestyle=(5,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'], color='red', linestyle=(10,(5,10)), lw=4)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label=f'UHI{suffix}_p2p_un_ucdb_sim', color='darkred', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label=f'UHI{suffix}_p2p_un_lcz_sim', color='peru', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label=f'UHI{suffix}_p2p_ms_ucdb_sim', color='red', lw=4)

ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label=f'UHI{suffix}_blend_ms_ucdb_sim', color='salmon', lw=4)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label=f'UHI{suffix}_blend_ms_ucdb_obs', color='salmon', linestyle='--', marker='o', lw=4)


ax.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')

ax.set_title(f"{city} UHI_{var} \n all methods ", fontsize=16)
ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI_{var} [°C]", fontsize=14)
ax.grid()
plt.legend()

fig.show()

In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label=f'UHI{suffix}_urclim_un_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label=f'UHI{suffix}_urclim_un_lcz_obs', color='peru', linestyle='--', marker='o', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label=f'UHI{suffix}_urclim_ms_ucdb_obs', color='red', linestyle='--', marker='o', lw=4)


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'], color='darkred', linestyle=(0,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'], color='peru', linestyle=(5,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'], color='red', linestyle=(10,(5,10)), lw=4)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label=f'UHI{suffix}_p2p_un_ucdb_sim', color='darkred', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label=f'UHI{suffix}_p2p_un_lcz_sim', color='peru', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label=f'UHI{suffix}_p2p_ms_ucdb_sim', color='red', lw=4)

ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label=f'UHI{suffix}_blend_ms_ucdb_sim', color='salmon', lw=4)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label=f'UHI{suffix}_blend_ms_ucdb_obs', color='salmon', linestyle='--', marker='o', lw=4)


ax.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')

ax.set_title(f"{city} UHI_{var}  \n Meteo station network comparison : P2P vs URCLIM vs BLEND ", fontsize=16)
ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI_{var} [°C]", fontsize=14)
ax.grid()
plt.legend()

fig.show()

In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label=f'UHI{suffix}_urclim_un_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label=f'UHI{suffix}_urclim_un_lcz_obs', color='peru', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label=f'UHI{suffix}_urclim_ms_ucdb_obs', color='red', linestyle='--', marker='o', lw=4, alpha = 0.2)


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'], color='darkred', linestyle=(0,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'], color='peru', linestyle=(5,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'], color='red', linestyle=(10,(5,10)), lw=4)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label=f'UHI{suffix}_p2p_un_ucdb_sim', color='darkred', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label=f'UHI{suffix}_p2p_un_lcz_sim', color='peru', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label=f'UHI{suffix}_p2p_ms_ucdb_sim', color='red', lw=4, alpha = 0.2)

ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label=f'UHI{suffix}_blend_ms_ucdb_sim', color='salmon', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label=f'UHI{suffix}_blend_ms_ucdb_obs', color='salmon', linestyle='--', marker='o', lw=4, alpha = 0.2)


ax.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')

ax.set_title(f"{city} UHI_{var} \n Urban network comparison : P2P vs URCLIM ", fontsize=16)
ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI_{var} [°C]", fontsize=14)
ax.grid()
plt.legend()

fig.show()

In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label=f'UHI{suffix}_urclim_un_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label=f'UHI{suffix}_urclim_un_lcz_obs', color='peru', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label=f'UHI{suffix}_urclim_ms_ucdb_obs', color='red', linestyle='--', marker='o', lw=4)


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'], color='darkred', linestyle=(0,(5,10)), lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'], color='peru', linestyle=(5,(5,10)), lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'], color='red', linestyle=(10,(5,10)), lw=4, alpha = 0.2)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label=f'UHI{suffix}_p2p_un_ucdb_sim', color='darkred', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label=f'UHI{suffix}_p2p_un_lcz_sim', color='peru', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label=f'UHI{suffix}_p2p_ms_ucdb_sim', color='red', lw=4)

ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label=f'UHI{suffix}_blend_ms_ucdb_sim', color='salmon', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label=f'UHI{suffix}_blend_ms_ucdb_obs', color='salmon', linestyle='--', marker='o', lw=4, alpha = 0.2)


ax.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')

ax.set_title(f"{city} UHI_{var} \n P2P comparison : UN_UCDB, UN_LCZ, MS_UCDB ", fontsize=16)
ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI_{var} [°C]", fontsize=14)
ax.grid()
plt.legend()

fig.show()

In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_obs'], label=f'UHI{suffix}_urclim_un_ucdb_obs', color='darkred', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_obs'], label=f'UHI{suffix}_urclim_un_lcz_obs', color='peru', linestyle='--', marker='o', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_obs'], label=f'UHI{suffix}_urclim_ms_ucdb_obs', color='red', linestyle='--', marker='o', lw=4)


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_sim'], color='darkred', linestyle=(0,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_sim'], color='peru', linestyle=(5,(5,10)), lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_sim'], color='red', linestyle=(10,(5,10)), lw=4)


ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_sim'], label=f'UHI{suffix}_p2p_un_ucdb_sim', color='darkred', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_sim'], label=f'UHI{suffix}_p2p_un_lcz_sim', color='peru', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_sim'], label=f'UHI{suffix}_p2p_ms_ucdb_sim', color='red', lw=4, alpha = 0.2)

ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_sim'], label=f'UHI{suffix}_blend_ms_ucdb_sim', color='salmon', lw=4, alpha = 0.2)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_obs'], label=f'UHI{suffix}_blend_ms_ucdb_obs', color='salmon', linestyle='--', marker='o', lw=4, alpha = 0.2)


ax.plot(np.zeros(12), color='green', lw=3,marker='o', label= 'rural_ref')

ax.set_title(f" {city} UHI_{var} \n URCLIM comparison : UN_UCDB, UN_LCZ, MS_UCDB ", fontsize=16)
ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI_{var} [°C]", fontsize=14)
ax.grid()
plt.legend()


In [ ]:

fig, ax = plt.subplots(figsize=(15, 7))


ax.plot(uhis[f'UHI{suffix}_urclim_un_ucdb_bias'], label=f'UHI{suffix}_urclim_un_ucdb_bias', color='darkred', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_un_lcz_bias'], label=f'UHI{suffix}_urclim_un_lcz_bias', color='peru', lw=4)
ax.plot(uhis[f'UHI{suffix}_urclim_ms_ucdb_bias'], label=f'UHI{suffix}_urclim_ms_ucdb_bias', color='crimson', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_un_ucdb_bias'], label=f'UHI{suffix}_p2p_un_ucdb_bias', color='firebrick', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_un_lcz_bias'], label=f'UHI{suffix}_p2p_un_lcz_bias', color='sandybrown', lw=4)
ax.plot(uhis[f'UHI{suffix}_p2p_ms_ucdb_bias'], label=f'UHI{suffix}_p2p_ms_ucdb_bias', color='red', lw=4)
ax.plot(uhis[f'UHI{suffix}_blend_ms_ucdb_bias'], label=f'UHI{suffix}_blend_ms_ucdb_bias', color='salmon', lw=4)


ax.plot(np.zeros(12), color='black', lw=3)
ax.set_title(f"{city} UHI_{var} bias \n all methods ", fontsize=16)

ax.set_xlabel("Month", fontsize=14)
#set x ticks to show month names
ax.set_xticks(np.arange(0, 12))
ax.set_xticklabels(['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'], fontsize=12)

ax.set_ylabel(f"UHI_{var} bias [°C]", fontsize=14)
ax.grid()
plt.legend()

In [ ]:
for uhi in uhis.data_vars:
    print(f"Variable: {uhi}, mean bias: {uhis[uhi][6:8].mean().item():.2f} °C")

In [ ]:
for var in biases.data_vars:
    if 'bias' in var:
       print(f"Variable: {var}, mean bias: {biases[var].mean().item():.2f} °C")
